# Week 5 Exercise: Retrieval-Augmented Generation

## Project Overview

For this assignment, I will build a RAG chatbot using the Wikipedia page for Cyberpunk 2077. The goal is to see whether the system can retrieve information from the article and use it to answer questions.

I chose Cyberpunk 2077 because I am familiar with the game and can evaluate whether the answers make sense. I will also look at how well the chatbot retrieves relevant information and whether it gives answers that are supported by the source.

In [1]:
%pip install -q requests beautifulsoup4

## Retrieving the Wikipedia Article

Before building the RAG system, I need to collect the information it will use to answer questions. I will retrieve the Cyberpunk 2077 Wikipedia article and extract its text using Beautiful Soup.

The article will serve as the knowledge source for the chatbot. I want to see whether the system can find relevant information from this source rather than simply generating answers from what the model already knows.

In [2]:
import requests
from bs4 import BeautifulSoup

url = "https://en.wikipedia.org/wiki/Cyberpunk_2077"

headers = {
    "User-Agent": "DSC670EducationalRAG/1.0 (student research project)"
}

response = requests.get(url, headers=headers, timeout=30)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

article = soup.select_one("#mw-content-text .mw-parser-output")

if article is None:
    raise ValueError("Wikipedia article content was not found.")

for element in article.select(
    "style, script, table, sup.reference, .mw-editsection, "
    ".navbox, .reflist, .hatnote"
):
    element.decompose()

paragraphs = [
    p.get_text(" ", strip=True)
    for p in article.find_all("p")
]

paragraphs = [p for p in paragraphs if len(p) > 40]

article_text = "\n\n".join(paragraphs)

print("Paragraphs collected:", len(paragraphs))
print("Total characters:", len(article_text))
print("\nPreview:\n")
print(article_text[:1200])


Paragraphs collected: 69
Total characters: 52775

Preview:

Cyberpunk 2077 is a 2020 action role-playing game developed by CD Projekt Red and published by CD Projekt . Based on Mike Pondsmith 's Cyberpunk tabletop game series, the plot is set in the fictional cyberpunk metropolis of Night City in California , within the dystopian Cyberpunk universe. The player assumes the role of V (voiced by Gavin Drea or Cherami Leigh depending on the player character 's gender), a mercenary who gets reluctantly imbued with a cybernetic "biochip" containing an engram of legendary rockstar and terrorist Johnny Silverhand (portrayed by Keanu Reeves ). As Johnny's consciousness begins overwriting V's own, the two must work together to separate from each other and save V's life by confronting the Arasaka Corporation, a powerful megacorporation acting as the de facto ruler of the city.

Cyberpunk 2077 began development following the release of The Witcher 3: Wild Hunt – Blood and Wine (2016). The game was

## Wikipedia Extraction Results

The Wikipedia extraction worked on the first attempt. I collected 69 paragraphs containing 52,775 characters from the Cyberpunk 2077 article. The preview included information about the game's development, which confirmed that the program was retrieving actual article content.

One thing I noticed was that the text was much longer than I expected. Instead of passing the entire article to the chatbot every time someone asks a question, I think it would make more sense to divide it into smaller sections. That way, the system can search for information related to a specific question without processing the whole article each time.

The extraction was successful, but I still need to test whether the chatbot can retrieve the correct information from the text.

In [3]:
chunk_size = 800
chunk_overlap = 150

chunks = []
start = 0

while start < len(article_text):
    end = min(start + chunk_size, len(article_text))
    chunk = article_text[start:end]
    chunks.append(chunk)

    if end == len(article_text):
        break

    start = end - chunk_overlap

print("Total chunks created:", len(chunks))
print("\nFirst chunk preview:\n")
print(chunks[0])


Total chunks created: 81

First chunk preview:

Cyberpunk 2077 is a 2020 action role-playing game developed by CD Projekt Red and published by CD Projekt . Based on Mike Pondsmith 's Cyberpunk tabletop game series, the plot is set in the fictional cyberpunk metropolis of Night City in California , within the dystopian Cyberpunk universe. The player assumes the role of V (voiced by Gavin Drea or Cherami Leigh depending on the player character 's gender), a mercenary who gets reluctantly imbued with a cybernetic "biochip" containing an engram of legendary rockstar and terrorist Johnny Silverhand (portrayed by Keanu Reeves ). As Johnny's consciousness begins overwriting V's own, the two must work together to separate from each other and save V's life by confronting the Arasaka Corporation, a powerful megacorporation acting as the de facto 


## Preparing the Text for Retrieval

The article was divided into 81 chunks using a maximum size of 800 characters and an overlap of 150 characters. The overlap allows some information to appear in neighboring chunks, which may help preserve context when a sentence falls near the end of a section.

The next step is to create embeddings for these chunks. Instead of searching only for matching words, embeddings will allow the system to compare the meaning of a question with the meaning of different sections of the article. I want to see whether this approach can retrieve useful information even when the question is worded differently from the Wikipedia text.

In [4]:
%pip install -q sentence-transformers

In [5]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_embeddings = embedding_model.encode(
    chunks,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Total chunks embedded:", len(chunk_embeddings))
print("Embedding dimensions:", chunk_embeddings.shape[1])
print("Embedding matrix shape:", chunk_embeddings.shape)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Total chunks embedded: 81
Embedding dimensions: 384
Embedding matrix shape: (81, 384)


## Testing Semantic Retrieval

The embedding model successfully converted all 81 chunks into numerical representations. The next step is to test whether the system can identify relevant sections of the article when I ask a question.

I want to test retrieval before adding the chatbot because finding the correct information is an important part of RAG. If the system retrieves unrelated text, the language model may have trouble producing an accurate answer even if the model itself works correctly.

In [6]:
import numpy as np

def retrieve_chunks(question, top_k=3):
    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    similarities = chunk_embeddings @ question_embedding
    top_indices = np.argsort(similarities)[::-1][:top_k]

    results = []

    for index in top_indices:
        results.append({
            "chunk_id": int(index),
            "similarity": float(similarities[index]),
            "text": chunks[index]
        })

    return results


question = "Who is Johnny Silverhand in Cyberpunk 2077?"

retrieved_results = retrieve_chunks(question)

print("Question:", question)

for result in retrieved_results:
    print("\nChunk:", result["chunk_id"])
    print("Similarity:", round(result["similarity"], 3))
    print("Text:", result["text"][:600])


Question: Who is Johnny Silverhand in Cyberpunk 2077?

Chunk: 0
Similarity: 0.629
Text: Cyberpunk 2077 is a 2020 action role-playing game developed by CD Projekt Red and published by CD Projekt . Based on Mike Pondsmith 's Cyberpunk tabletop game series, the plot is set in the fictional cyberpunk metropolis of Night City in California , within the dystopian Cyberpunk universe. The player assumes the role of V (voiced by Gavin Drea or Cherami Leigh depending on the player character 's gender), a mercenary who gets reluctantly imbued with a cybernetic "biochip" containing an engram of legendary rockstar and terrorist Johnny Silverhand (portrayed by Keanu Reeves ). As Johnny's consc

Chunk: 13
Similarity: 0.606
Text:  utilise paramilitary doctors for rapid medical services. Due to the constant threat of physical harm, open carry is permitted for all residents of Night City.

The primary protagonist of Cyberpunk 2077 is Vincent / Valerie, known as V ( Gavin Drea or Cherami Leigh ), a freel

## Semantic Retrieval Results

For the first retrieval test, I asked who Johnny Silverhand is in Cyberpunk 2077. The system returned three chunks with similarity scores of 0.629, 0.606, and 0.554.

At first, I thought the highest-ranked chunk was less relevant because I was only looking at the first 600 characters. After displaying the full passages, I realized that Chunk 0 actually contained important information about Johnny Silverhand and his connection to V through the biochip. Chunk 13 was also useful because it described Johnny as a rock star and a digitized engram.

Chunk 40 was somewhat different. It included information about Keanu Reeves portraying Johnny Silverhand, which was related to the character but did not directly explain who he was in the story.

I think the retrieval worked well for this question. It also showed me why reviewing the complete text matters. My first impression of the results changed once I could see all the information that had been retrieved.

In [7]:
for result in retrieved_results:
    print("=" * 70)
    print("Chunk:", result["chunk_id"])
    print("Similarity:", round(result["similarity"], 3))
    print("\nFull retrieved text:\n")
    print(result["text"])
    print()


Chunk: 0
Similarity: 0.629

Full retrieved text:

Cyberpunk 2077 is a 2020 action role-playing game developed by CD Projekt Red and published by CD Projekt . Based on Mike Pondsmith 's Cyberpunk tabletop game series, the plot is set in the fictional cyberpunk metropolis of Night City in California , within the dystopian Cyberpunk universe. The player assumes the role of V (voiced by Gavin Drea or Cherami Leigh depending on the player character 's gender), a mercenary who gets reluctantly imbued with a cybernetic "biochip" containing an engram of legendary rockstar and terrorist Johnny Silverhand (portrayed by Keanu Reeves ). As Johnny's consciousness begins overwriting V's own, the two must work together to separate from each other and save V's life by confronting the Arasaka Corporation, a powerful megacorporation acting as the de facto 

Chunk: 13
Similarity: 0.606

Full retrieved text:

 utilise paramilitary doctors for rapid medical services. Due to the constant threat of physical 

## Generating Answers With Retrieved Information

So far, the system has been able to find relevant passages from the Wikipedia article, but it has not generated an answer. The next step is to connect the retriever to a language model.

I will use a Hugging Face text-generation model and provide it with the retrieved passages as context. My goal is to see whether the model can answer a question using the information from Wikipedia rather than relying entirely on what it learned during training.

I also want to see whether the generated answer accurately reflects the retrieved text. A successful retrieval does not automatically mean the final answer will be correct.

In [8]:
%pip install -q transformers sentencepiece


In [9]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

generation_model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(generation_model_name)

generation_model = AutoModelForSeq2SeqLM.from_pretrained(
    generation_model_name
)

print("Answer-generation model loaded successfully.")


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Answer-generation model loaded successfully.


In [10]:
def ask_rag(question, top_k=3):
    results = retrieve_chunks(question, top_k=top_k)

    context = "\n\n".join(
        result["text"] for result in results
    )

    prompt = f"""Answer the question using only the context below.
If the answer is not in the context, say "I could not find that information in the article."
Keep the answer brief.

Context:
{context}

Question: {question}

Answer:"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    outputs = generation_model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer


question = "Who is Johnny Silverhand in Cyberpunk 2077?"

answer = ask_rag(question)

print("Question:", question)
print("RAG Answer:", answer)


Question: Who is Johnny Silverhand in Cyberpunk 2077?
RAG Answer: I could not find that information in the article.


## Initial RAG Answer Evaluation

The first complete RAG test did not produce the answer I expected. When I asked who Johnny Silverhand was, the model responded that it could not find the information in the article.

This was surprising because the retrieval stage had already identified passages containing information about Johnny Silverhand. The issue may be happening during answer generation rather than retrieval.

One possible explanation is that the model is receiving more text than its input limit allows. I will check the number of tokens being passed to the model before deciding what needs to be changed.

In [11]:
results = retrieve_chunks(question, top_k=3)

context = "\n\n".join(
    result["text"] for result in results
)

prompt = f"""Answer the question using only the context below.
If the answer is not in the context, say "I could not find that information in the article."
Keep the answer brief.

Context:
{context}

Question: {question}

Answer:"""

tokens = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=False
)

print("Input tokens:", tokens["input_ids"].shape[1])
print("Current token limit:", 512)
print("Would be truncated:", tokens["input_ids"].shape[1] > 512)


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (657 > 512). Running this sequence through the model will result in indexing errors


Input tokens: 657
Current token limit: 512
Would be truncated: True


In [12]:
def ask_rag_v2(question):
    results = retrieve_chunks(question, top_k=3)

    best_result = max(
        results,
        key=lambda result: result["similarity"]
    )

    context = best_result["text"]

    prompt = f"""Read the passage and answer the question.

Passage:
{context}

Question: {question}

Give a short answer based only on the passage.
If the passage does not contain the answer, say "Not found."

Answer:"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    print("Tokens sent to model:", inputs["input_ids"].shape[1])
    print("Retrieved chunk:", best_result["chunk_id"])

    outputs = generation_model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer


question = "Who is Johnny Silverhand in Cyberpunk 2077?"

answer_v2 = ask_rag_v2(question)

print("\nQuestion:", question)
print("Revised RAG Answer:", answer_v2)


Tokens sent to model: 260
Retrieved chunk: 0

Question: Who is Johnny Silverhand in Cyberpunk 2077?
Revised RAG Answer: legendary rockstar and terrorist


## Investigating the Input-Length Problem

The token-count test showed that the original prompt contained 657 tokens, while the model was configured to accept only 512. This meant that 145 tokens would be removed before the model generated its answer.

This gave me a possible explanation for the unsuccessful response. Even though the retriever found information about Johnny Silverhand, the language model may not have received all of it.

For the next test, I changed the system to send only the highest-ranked chunk instead of combining three chunks. This should keep the input shorter, although it also means the model will have less information available. I want to compare the new response with the original result before deciding whether this is a better approach.

## Revised RAG Results

The revised version produced a much better response. Instead of saying the information was unavailable, the model identified Johnny Silverhand as a "legendary rockstar and terrorist." This matched the information in the Wikipedia passages retrieved earlier.

The biggest difference was the amount of text sent to the model. The original prompt contained 657 tokens and exceeded the 512-token limit. After changing the system to use only the highest-ranked chunk, the input dropped to 260 tokens.

I think this was a useful improvement because the model was finally able to answer the question using the article. However, I also recognize that using only one chunk could become a problem if the answer requires information from multiple sections. I would need to test additional questions before deciding whether this approach is reliable.

In [13]:
test_questions = [
    "Where does Cyberpunk 2077 take place?",
    "Who developed Cyberpunk 2077?",
    "What is the name of the main playable character?",
    "Who portrays Johnny Silverhand?",
    "What is Phantom Liberty?"
]

for test_question in test_questions:
    print("=" * 65)
    print("Question:", test_question)

    test_answer = ask_rag_v2(test_question)

    print("Answer:", test_answer)
    print()


Question: Where does Cyberpunk 2077 take place?
Tokens sent to model: 234
Retrieved chunk: 11
Answer: Night City (NC)

Question: Who developed Cyberpunk 2077?
Tokens sent to model: 255
Retrieved chunk: 24
Answer: Pondsmith

Question: What is the name of the main playable character?
Tokens sent to model: 317
Retrieved chunk: 15
Answer: Not found

Question: Who portrays Johnny Silverhand?
Tokens sent to model: 255
Retrieved chunk: 0
Answer: Keanu Reeves

Question: What is Phantom Liberty?
Tokens sent to model: 231
Retrieved chunk: 5
Answer: Not found



## Evaluating Multiple RAG Questions

I tested the revised chatbot using five questions about different parts of Cyberpunk 2077. It correctly identified Night City as the setting and Keanu Reeves as the actor portraying Johnny Silverhand.

The other three responses were less successful. When I asked who developed the game, the model answered Pondsmith instead of CD Projekt Red. It also failed to identify V as the main playable character and could not explain what Phantom Liberty was.

Based on these results, the revised system answered two of the five questions correctly. I think this shows that shortening the input helped with the original Johnny Silverhand question, but it did not make the chatbot consistently accurate. I want to inspect the retrieved passages for the unsuccessful questions to determine whether the problem is with retrieval or answer generation.

In [14]:
failed_questions = [
    "Who developed Cyberpunk 2077?",
    "What is the name of the main playable character?",
    "What is Phantom Liberty?"
]

for test_question in failed_questions:
    print("=" * 70)
    print("Question:", test_question)

    results = retrieve_chunks(test_question, top_k=3)

    for result in results:
        print("\nChunk:", result["chunk_id"])
        print("Similarity:", round(result["similarity"], 3))
        print("Text:", result["text"])
        print()

Question: Who developed Cyberpunk 2077?

Chunk: 24
Similarity: 0.679
Text: ine (2005), provided valuable wisdom to CD Projekt Red, in comparison to Polish writer Andrzej Sapkowski 's indifference towards the studio during the development of The Witcher (2007) and The Witcher 2 .

The concept of Cyberpunk 2077 was adapted from Pondsmith's Cyberpunk series. Antithetical to the stereotypical cyberpunk genre, the concepts explored in Cyberpunk evoke a sense of rebellion and whimsy while retaining the genre's grim tone.  The protagonist of Cyberpunk —unlike the typical archetype of a hero attempting to save the world—attempts to save themself from a downtrodden perspective. Lead gameplay designer Marcin Janiszewski sought to remind players of its connection to the Cyberpunk universe, writing, "We want to assure fans of the pen-and-paper game that this is still the sam


Chunk: 44
Similarity: 0.678
Text: ity outdoors and in cinemas, using established brands to disseminate information about t

## Testing Answer Generation With Better Context

After reviewing the unsuccessful questions, I noticed that the retriever was finding some of the correct information. For example, the passages mentioned CD Projekt Red, V, and Phantom Liberty, but the chatbot still returned incorrect answers or said the information was unavailable.

This made me think the problem was not entirely with retrieval. The model might be struggling to identify the most useful sentence within a longer passage.

For the next experiment, I want to test the language model using shorter passages that contain the answers. This will help me determine whether the model performs better when it receives more focused information.

In [15]:
focused_tests = [
    {
        "question": "Who developed Cyberpunk 2077?",
        "context": "Cyberpunk 2077 is a 2020 action role-playing game developed by CD Projekt Red and published by CD Projekt."
    },
    {
        "question": "What is the name of the main playable character?",
        "context": "The player assumes the role of V, a mercenary in Night City."
    },
    {
        "question": "What is Phantom Liberty?",
        "context": "Phantom Liberty is a downloadable content expansion for Cyberpunk 2077."
    }
]

for test in focused_tests:
    prompt = f"""Answer the question using the passage.

Passage: {test['context']}

Question: {test['question']}

Answer:"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    outputs = generation_model.generate(
        **inputs,
        max_new_tokens=60,
        do_sample=False
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    print("Question:", test["question"])
    print("Answer:", answer)
    print("-" * 60)


Question: Who developed Cyberpunk 2077?
Answer: CD Projekt Red
------------------------------------------------------------
Question: What is the name of the main playable character?
Answer: V
------------------------------------------------------------
Question: What is Phantom Liberty?
Answer: downloadable content expansion for Cyberpunk 2077
------------------------------------------------------------


## Focused Context Test Results

The focused context test worked much better than the earlier chatbot responses. When I provided shorter passages, the model correctly identified CD Projekt Red as the developer, V as the main playable character, and Phantom Liberty as an expansion for Cyberpunk 2077.

What I found interesting was that I did not have to change the language model to get these answers. I only changed the information being provided to it. This suggests that the longer passages may have included too much unrelated information, making it harder for the model to identify what was important.

However, these were manually selected passages, so I would not consider this proof that the RAG system itself has improved. The next step is to see whether I can automatically select more relevant information from the Wikipedia article and achieve similar results.

In [16]:
import re

def retrieve_focused_context(question, top_k=3, max_sentences=3):
    results = retrieve_chunks(question, top_k=top_k)

    sentences = []

    for result in results:
        parts = re.split(r'(?<=[.!?])\s+', result["text"])

        for part in parts:
            part = part.strip()

            if len(part) > 25:
                sentences.append(part)

    sentence_embeddings = embedding_model.encode(
        sentences,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    similarities = sentence_embeddings @ question_embedding
    top_indices = np.argsort(similarities)[::-1][:max_sentences]

    focused_context = "\n".join(
        sentences[index] for index in top_indices
    )

    return focused_context


def ask_rag_v3(question):
    context = retrieve_focused_context(question)

    prompt = f"""Answer the question using only the information below.
If the information does not contain the answer, say "Not found."

Information:
{context}

Question: {question}

Answer:"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    outputs = generation_model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer


for test_question in test_questions:
    print("=" * 65)
    print("Question:", test_question)
    print("Answer:", ask_rag_v3(test_question))
    print()

Question: Where does Cyberpunk 2077 take place?
Answer: Night City (NC)

Question: Who developed Cyberpunk 2077?
Answer: Pondsmith

Question: What is the name of the main playable character?
Answer: Johnny Silverhand

Question: Who portrays Johnny Silverhand?
Answer: Keanu Reeves

Question: What is Phantom Liberty?
Answer: A DLC expansion



In [17]:
diagnostic_questions = [
    "Who developed Cyberpunk 2077?",
    "What is the name of the main playable character?"
]

for test_question in diagnostic_questions:
    print("=" * 70)
    print("Question:", test_question)
    print("\nSelected context:\n")
    print(retrieve_focused_context(test_question))
    print()

Question: Who developed Cyberpunk 2077?

Selected context:

Cyberpunk 2077 was announced in May 2012.
The events of Cyberpunk 2077 take place in the fictitious N
The concept of Cyberpunk 2077 was adapted from Pondsmith's Cyberpunk series.

Question: What is the name of the main playable character?

Selected context:

The player assumes the role of V (voiced by Gavin Drea or Cherami Leigh depending on the player character 's gender), a mercenary who gets reluctantly imbued with a cybernetic "biochip" containing an engram of legendary rockstar and terrorist Johnny Silverhand (portrayed by Keanu Reeves ).
Other sources of inspiration for the team include the manga and anime series Ghost in the Shell , and other video games such as System Shock (1994) and the first part of Deus Ex (2000).
The game opens with one of three lifepaths for the player character V—Nomad, Streetkid, or Corpo—each leading



## Evaluating the Revised Retrieval Method

The third version produced mixed results. It correctly identified Night City, Keanu Reeves, and Phantom Liberty, but it still confused Mike Pondsmith with the game's developer. It also identified Johnny Silverhand as the main playable character instead of V.

I expected selecting shorter sentences to improve the answers, especially after the focused context test worked so well. However, the results showed that shorter context does not necessarily mean better context. The system still needs to select information that directly answers the question.

The most concerning result was the incorrect playable character. Version 2 said the information was not found, while Version 3 confidently provided the wrong character. That would be a bigger problem in a real application because someone might trust an incorrect answer without realizing the system made a mistake.

I want to inspect the selected sentences before making another adjustment. This should help me understand whether the problem is happening during retrieval or when the language model interprets the retrieved text.

In [18]:
def retrieve_focused_context_v2(question, max_sentences=3):
    sentences = []

    for chunk in chunks:
        parts = re.split(r'(?<=[.!?])\s+', chunk)

        for part in parts:
            part = part.strip()

            if len(part) > 25:
                sentences.append(part)

    sentence_embeddings = embedding_model.encode(
        sentences,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    similarities = sentence_embeddings @ question_embedding
    top_indices = np.argsort(similarities)[::-1][:max_sentences]

    return "\n".join(sentences[index] for index in top_indices)


for test_question in diagnostic_questions:
    print("=" * 70)
    print("Question:", test_question)
    print("\nNew selected context:\n")
    print(retrieve_focused_context_v2(test_question))
    print()


Question: Who developed Cyberpunk 2077?

New selected context:

The versions of Cyberpunk 2077
Cyberpunk 2077 takes place
Cyberpunk 2077 is a 2020 action role-playing game developed by CD Projekt Red and published by CD Projekt .

Question: What is the name of the main playable character?

New selected context:

The player assumes the role of V (voiced by Gavin Drea or Cherami Leigh depending on the player character 's gender), a mercenary who gets reluctantly imbued with a cybernetic "biochip" containing an engram of legendary rockstar and terrorist Johnny Silverhand (portrayed by Keanu Reeves ).
V is accompanied by Johnny Silverhand ( Keanu Reeves ), a legendary rock star and terrorist who now exists as a digitised engram; Johnny is also playable during some segments of the game.
Talsorian Games , in early 2012, sending him a copy of The Witcher 2: Assassins of Kings (2011).



In [19]:
def ask_rag_v4(question):
    context = retrieve_focused_context_v2(
        question,
        max_sentences=3
    )

    prompt = f"""Use the information below to answer the question.
Answer briefly and use only the provided information.
If the answer is unavailable, say "Not found."

Information:
{context}

Question: {question}

Answer:"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    outputs = generation_model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer


for test_question in test_questions:
    print("=" * 65)
    print("Question:", test_question)
    print("Answer:", ask_rag_v4(test_question))
    print()


Question: Where does Cyberpunk 2077 take place?
Answer: fictitious N

Question: Who developed Cyberpunk 2077?
Answer: CD Projekt Red

Question: What is the name of the main playable character?
Answer: Johnny Silverhand

Question: Who portrays Johnny Silverhand?
Answer: Keanu Reeves

Question: What is Phantom Liberty?
Answer: DLC expansion



In [20]:
import re

clean_sentences = []

for paragraph in paragraphs:
    parts = re.split(r'(?<=[.!?])\s+', paragraph)

    for sentence in parts:
        sentence = sentence.strip()

        if len(sentence) >= 40:
            clean_sentences.append(sentence)

print("Total complete sentences:", len(clean_sentences))
print("\nFirst five sentences:\n")

for sentence in clean_sentences[:5]:
    print(sentence)
    print()

Total complete sentences: 342

First five sentences:

Cyberpunk 2077 is a 2020 action role-playing game developed by CD Projekt Red and published by CD Projekt .

Based on Mike Pondsmith 's Cyberpunk tabletop game series, the plot is set in the fictional cyberpunk metropolis of Night City in California , within the dystopian Cyberpunk universe.

The player assumes the role of V (voiced by Gavin Drea or Cherami Leigh depending on the player character 's gender), a mercenary who gets reluctantly imbued with a cybernetic "biochip" containing an engram of legendary rockstar and terrorist Johnny Silverhand (portrayed by Keanu Reeves ).

As Johnny's consciousness begins overwriting V's own, the two must work together to separate from each other and save V's life by confronting the Arasaka Corporation, a powerful megacorporation acting as the de facto ruler of the city.

Cyberpunk 2077 began development following the release of The Witcher 3: Wild Hunt – Blood and Wine (2016).



In [21]:
sentence_embeddings = embedding_model.encode(
    clean_sentences,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Total sentences embedded:", len(sentence_embeddings))
print("Embedding dimensions:", sentence_embeddings.shape[1])
print("Embedding matrix shape:", sentence_embeddings.shape)


Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Total sentences embedded: 342
Embedding dimensions: 384
Embedding matrix shape: (342, 384)


In [22]:
def retrieve_clean_sentences(question, top_k=3):
    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    similarities = sentence_embeddings @ question_embedding
    top_indices = np.argsort(similarities)[::-1][:top_k]

    results = []

    for index in top_indices:
        results.append({
            "sentence_id": int(index),
            "similarity": float(similarities[index]),
            "text": clean_sentences[index]
        })

    return results


test_questions = [
    "Where does Cyberpunk 2077 take place?",
    "Who developed Cyberpunk 2077?",
    "What is the name of the main playable character?",
    "Who portrays Johnny Silverhand?",
    "What is Phantom Liberty?"
]

for question in test_questions:
    print("=" * 70)
    print("Question:", question)

    results = retrieve_clean_sentences(question, top_k=3)

    for result in results:
        print("\nSimilarity:", round(result["similarity"], 3))
        print("Sentence:", result["text"])

    print()


Question: Where does Cyberpunk 2077 take place?

Similarity: 0.825
Sentence: Cyberpunk 2077 was announced in May 2012.

Similarity: 0.791
Sentence: Cyberpunk 2077 is a 2020 action role-playing game developed by CD Projekt Red and published by CD Projekt .

Similarity: 0.784
Sentence: A sequel to Cyberpunk 2077 was announced in October 2022.

Question: Who developed Cyberpunk 2077?

Similarity: 0.817
Sentence: Cyberpunk 2077 is a 2020 action role-playing game developed by CD Projekt Red and published by CD Projekt .

Similarity: 0.799
Sentence: Cyberpunk 2077 was announced in May 2012.

Similarity: 0.787
Sentence: The concept of Cyberpunk 2077 was adapted from Pondsmith's Cyberpunk series.

Question: What is the name of the main playable character?

Similarity: 0.496
Sentence: The player assumes the role of V (voiced by Gavin Drea or Cherami Leigh depending on the player character 's gender), a mercenary who gets reluctantly imbued with a cybernetic "biochip" containing an engram of leg

In [23]:
def ask_rag_v5(question):
    results = retrieve_clean_sentences(question, top_k=3)

    context = "\n".join(
        result["text"] for result in results
    )

    prompt = f"""Answer the question using only the information below.
If the answer cannot be found, say "Not found."
Give a short, direct answer.

Information:
{context}

Question: {question}

Answer:"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    outputs = generation_model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer


for test_question in test_questions:
    print("=" * 65)
    print("Question:", test_question)
    print("Answer:", ask_rag_v5(test_question))
    print()


Question: Where does Cyberpunk 2077 take place?
Answer: Not found.

Question: Who developed Cyberpunk 2077?
Answer: CD Projekt Red

Question: What is the name of the main playable character?
Answer: Johnny Silverhand

Question: Who portrays Johnny Silverhand?
Answer: Keanu Reeves

Question: What is Phantom Liberty?
Answer: DLC expansion



## Interactive RAG Chatbot

After testing several retrieval approaches, I selected the sentence-based version for the final chatbot. It uses the Cyberpunk 2077 Wikipedia article as its knowledge source, retrieves relevant sentences using embeddings, and generates responses with a Hugging Face language model.

The previous tests showed that the system can answer some questions correctly but can also miss relevant information or confuse characters. I decided to keep the chatbot interactive so I could test additional questions and observe how it performs beyond the original five examples.

In [25]:
print("Cyberpunk 2077 RAG Chatbot")
print("Ask a question about the game.")
print("Type 'quit' to end the conversation.\n")

while True:
    user_question = input("You: ").strip()

    if user_question.lower() in ["quit", "exit"]:
        print("Chatbot: Conversation ended.")
        break

    if not user_question:
        print("Chatbot: Please enter a question.\n")
        continue

    chatbot_answer = ask_rag_v5(user_question)

    print("Chatbot:", chatbot_answer)
    print()

Cyberpunk 2077 RAG Chatbot
Ask a question about the game.
Type 'quit' to end the conversation.

You: Who is Johnny Silverhand?
Chatbot: legendary rock star and terrorist

You: Who is V?
Chatbot: Johnny Silverhand

You: What company developed Cyberpunk 2077?
Chatbot: CD Projekt Red

You: Where does Cyberpunk 2077 take place?
Chatbot: Not found.

You: Where does Cyberpunk 2077 take place?
Chatbot: Not found.

You: quit
Chatbot: Conversation ended.


## Interactive Chatbot Evaluation

After building the interactive chatbot, I tested it with questions about Cyberpunk 2077. The chatbot correctly described Johnny Silverhand as a legendary rock star and terrorist. It also identified CD Projekt Red as the company that developed the game.

However, the chatbot struggled when I asked where Cyberpunk 2077 takes place. It responded with "Not found," even though the Wikipedia article identifies Night City as the game's setting. I asked the same question twice and received the same response.

I found this interesting because the answer was available in the source material, but the system still failed to retrieve it. This reinforced something I noticed throughout the assignment: having the correct information in a dataset does not guarantee that a RAG chatbot will find or use it correctly.

The interactive test also helped me see the difference between a chatbot that functions technically and one that consistently provides reliable answers. The system successfully accepted questions, retrieved information, and generated responses, but its accuracy depended on the information selected during retrieval.

## Final Thoughts and Conclusion

Prior to taking on this assignment, I understood the basic idea behind RAG but I’d never gone through the process of building one myself. I chose the Wikipedia page for Cyberpunk 2077 because I was already familiar with the game, and thought it would be easier to pick up when the chatbot gives an incorrect answer. I used Beautiful Soup to pull the information from Wikipedia and Hugging Face models to help the chatbot find information and answer questions.

I ran into more problems than I expected. There were times when the chatbot would retrieve information that seemed relevant but still give the wrong answer. One example was when it identified Johnny Silverhand as the main playable character instead of V. It also confused Mike Pondsmith, who created the original Cyberpunk tabletop game, with CD Projekt Red, the company that developed Cyberpunk 2077. What made this interesting was that the correct information was already in the article.

I tried a few different things to try and figure out what was happening. I even fed the language model shorter passages that had the answers, and it got all three questions right. So I thought maybe the model was the real issue or maybe it was the information going into the model that made it harder. I also noticed that some of my original text chunks were cutting sentences in half, so I changed the retrieval process to work with complete sentences instead.

Even after making those changes, the final chatbot still had limitations. It answered three out of five evaluation questions correctly, but it could not identify Night City as the setting when I tested the interactive chat. I thought that was a good example of how a system can have access to information without necessarily retrieving the right part of it.

Looking back at the different experiments, I think the troubleshooting was probably the most useful part of this assignment. I learned that improving a RAG system involves more than changing the language model or adding more information. The way the text is prepared, retrieved, and presented can make a difference in the answers. If I continued working on this project, I would focus on improving the retrieval rankings and testing a larger variety of questions before considering the chatbot reliable.